# Local affine and TPS refinement


In [ ]:
from __future__ import annotations
from pathlib import Path
import os
import sys

ROOT = next((path for path in (Path.cwd(), *Path.cwd().parents)
             if (path / "config" / "project.yaml").is_file()
             and (path / "src" / "pathology").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError("Start Jupyter within this repository.")
sys.path.insert(0, str(ROOT / "src"))
from pathology.config import load_paths
PATHS = load_paths(ROOT)
os.environ.setdefault("OMP_NUM_THREADS", "8")
os.environ.setdefault("LOKY_MAX_CPU_COUNT", "8")
os.environ.setdefault("HF_HOME", str(ROOT / ".cache" / "huggingface"))
os.environ.setdefault("HF_MODULES_CACHE", str(ROOT / ".cache" / "huggingface" / "modules"))
os.environ.setdefault("TORCH_HOME", str(ROOT / ".cache" / "torch"))


## Imports


In [ ]:
import gc
import json
import math
import os
import re
import time
import traceback
from contextlib import nullcontext
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Any, Optional
QPTIFF_CACHE_ROOT = ROOT / '.cache' / 'registration'
QPTIFF_CACHE_ROOT.mkdir(parents=True, exist_ok=True)
os.environ.setdefault('HF_HOME', str(QPTIFF_CACHE_ROOT / 'huggingface'))
os.environ.setdefault('HF_HUB_CACHE', str(QPTIFF_CACHE_ROOT / 'huggingface' / 'hub'))
os.environ.setdefault('TRANSFORMERS_CACHE', str(QPTIFF_CACHE_ROOT / 'huggingface' / 'transformers'))
os.environ.setdefault('TORCH_HOME', str(QPTIFF_CACHE_ROOT / 'torch'))
import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tifffile
import torch
import torchvision
import transformers
from PIL import Image
from scipy.interpolate import RBFInterpolator
from tqdm.auto import tqdm
from transformers import AutoImageProcessor, AutoModelForKeypointMatching
Image.MAX_IMAGE_PIXELS = None
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


## Configuration


In [ ]:
@dataclass
class Config:
    HE_DIR: str = str(PATHS.he_dir)
    MIF_DIR: str = str(PATHS.mif_dir)
    V3_RESULTS_DIR: str = str(PATHS.registration_global)
    OUTPUT_DIR: str = str(PATHS.registration)
    EXTENSIONS: tuple[str, ...] = ('.qptiff', '.qptif')
    CASE_ID_REGEX: str = '^(?P<case_id>.+?)_Scan\\d+$'
    RUN_CASE_IDS: tuple[str, ...] = ()
    PYRAMID_TARGET_LONG_EDGE: int = 3000
    PYRAMID_MAX_LONG_EDGE: int = 3500
    MIF_DAPI_CHANNEL: Optional[int] = None
    MIF_AUTO_TOP_CHANNELS: int = 2
    MIF_PERCENTILES: tuple[float, float] = (0.5, 99.8)
    MODEL_ID: str = 'zju-community/matchanything_eloftr'
    MODEL_CANVAS: int = 1024
    MATCH_MIN_SCORE: float = 0.1
    SCORE_THRESHOLDS: tuple[float, ...] = (0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 0.95)
    USE_MIXED_PRECISION: bool = True
    TILE_MODEL_CANVAS: int = 768
    RELOAD_MODEL_EVERY_CASE: bool = True
    RELOAD_MODEL_EVERY_N_CALLS: int = 48
    AUTO_RELOAD_AFTER_EMPTY_TILES: int = 2
    MAX_EMPTY_RELOADS_PER_CASE: int = 1
    TILE_RAW_MATCH_THRESHOLD: float = 0.03
    ENABLE_SIFT_FALLBACK: bool = True
    SIFT_NFEATURES: int = 5000
    SIFT_RATIO_TEST: float = 0.78
    SIFT_MIN_MATCHES: int = 8
    USE_BIDIRECTIONAL_MATCHING: bool = True
    MUTUAL_TOLERANCE_CANVAS_PX: float = 12.0
    MIN_MUTUAL_MATCHES: int = 12
    GRID_SIZE: int = 8
    MAX_MATCHES_PER_GRID_CELL: int = 24
    MAX_RANSAC_CANDIDATES: int = 1500
    ENABLE_ORIENTATION_RESCUE: bool = False
    RESCUE_ORIENTATIONS: tuple[str, ...] = ('rot90', 'rot180', 'rot270', 'flip_h', 'flip_v')
    RANSAC_THRESHOLD_PX: float = 5.0
    RANSAC_CONFIDENCE: float = 0.999
    RANSAC_MAX_ITERS: int = 30000
    ALLOW_AFFINE: bool = True
    MIN_SCALE: float = 0.25
    MAX_SCALE: float = 4.0
    MAX_ANISOTROPY: float = 1.35
    MAX_TRANSLATION_FRACTION: float = 1.5
    ALLOW_REFLECTION: bool = False
    HARD_MIN_INLIERS: int = 12
    FAIL_MIN_INLIER_RATIO: float = 0.1
    FAIL_MIN_SPATIAL_COVERAGE: float = 0.005
    FAST_ACCEPT_MIN_INLIERS: int = 40
    FAST_ACCEPT_MIN_RATIO: float = 0.5
    FAST_ACCEPT_MIN_COVERAGE: float = 0.05
    QC_MIN_INLIERS: int = 30
    QC_MIN_INLIER_RATIO: float = 0.5
    QC_MIN_SPATIAL_COVERAGE: float = 0.03
    REUSE_V3_GLOBAL: bool = True
    TILE_SIZE: int = 768
    TILE_STRIDE: int = 512
    MAX_TILES: int = 16
    TILE_MIN_OVERLAP_FRACTION: float = 0.05
    TILE_MATCH_SCORE: float = 0.25
    TILE_GUIDED_MAX_RESIDUAL_PX: float = 64.0
    TILE_RANSAC_THRESHOLD_PX: float = 3.0
    TILE_MIN_INLIERS: int = 6
    MIN_ACCEPTED_TILES: int = 4
    MIN_AGGREGATED_CONTROLS: int = 40
    CONTROL_DEDUP_BIN_PX: float = 12.0
    MAX_CONTROL_POINTS: int = 400
    MATCH_QC_MAX_LINES: int = 180
    MATCH_QC_PANEL_HEIGHT: int = 1300
    MATCH_QC_GAP_PX: int = 24
    MATCH_QC_LINE_ALPHA: float = 0.72
    MATCH_QC_RANDOM_SEED: int = 6211
    REFINE_RANSAC_THRESHOLD_PX: float = 3.0
    MAX_AFFINE_UPDATE_PX: float = 160.0
    TPS_MIN_CONTROLS: int = 24
    TPS_SMOOTHING: float = 12.0
    TPS_NEIGHBORS: int = 64
    TPS_MAX_DISPLACEMENT_PX: float = 96.0
    FIELD_GRID_SPACING: int = 32
    MAX_FOLD_FRACTION: float = 0.01
    MAX_QC_MATCHES: int = 300
    RANDOM_SEED: int = 2026
CFG = Config()
HE_DIR = Path(CFG.HE_DIR).resolve()
MIF_DIR = Path(CFG.MIF_DIR).resolve()
V3_RESULTS_DIR = Path(CFG.V3_RESULTS_DIR).resolve()
OUTPUT_DIR = Path(CFG.OUTPUT_DIR).resolve()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / 'pairs').mkdir(parents=True, exist_ok=True)
assert CFG.MODEL_CANVAS % 32 == 0
assert CFG.TILE_SIZE >= 256
assert CFG.TILE_STRIDE <= CFG.TILE_SIZE
assert CFG.TPS_MIN_CONTROLS >= 6
np.random.seed(CFG.RANDOM_SEED)
torch.manual_seed(CFG.RANDOM_SEED)


## Pyramid reader


In [ ]:
def _yx_shape(shape: tuple[int, ...], axes: str) -> tuple[int, int]:
    axes = axes.upper()
    if 'Y' not in axes or 'X' not in axes:
        raise ValueError(f'No Y/X axes: axes={axes!r}, shape={shape}')
    return (int(shape[axes.index('Y')]), int(shape[axes.index('X')]))

def _select_main_series(tif: tifffile.TiffFile) -> int:
    candidates = []
    for i, series in enumerate(tif.series):
        try:
            h, w = _yx_shape(tuple(series.shape), series.axes)
            candidates.append((h * w, i))
        except Exception:
            pass
    if not candidates:
        raise ValueError('No TIFF series contains X/Y axes')
    return max(candidates)[1]

def inspect_qptiff(path: Path) -> dict[str, Any]:
    with tifffile.TiffFile(path) as tif:
        series_index = _select_main_series(tif)
        series = tif.series[series_index]
        levels = list(getattr(series, 'levels', [series]))
        infos = []
        for i, level in enumerate(levels):
            h, w = _yx_shape(tuple(level.shape), level.axes)
            infos.append({'level': i, 'height': h, 'width': w, 'axes': level.axes, 'shape': tuple((int(x) for x in level.shape))})
        return {'path': str(path), 'name': path.name, 'series_index': series_index, 'full_height': infos[0]['height'], 'full_width': infos[0]['width'], 'levels': infos}

def _choose_level(levels: list[dict[str, Any]], target_long_edge: int) -> int:
    target = max(1, int(target_long_edge))
    return min(((abs(math.log(max(max(x['height'], x['width']), 1) / target)), x['level']) for x in levels))[1]

def _reduce_to_yxc(arr: np.ndarray, axes: str) -> tuple[np.ndarray, list[str]]:
    a = np.asarray(arr)
    labels = list(axes.upper())
    for i in range(a.ndim - 1, -1, -1):
        if a.shape[i] == 1:
            a = np.take(a, 0, axis=i)
            labels.pop(i)
    channel_label = 'C' if 'C' in labels else 'S' if 'S' in labels else None
    keep = {'Y', 'X'} | ({channel_label} if channel_label else set())
    for i in range(len(labels) - 1, -1, -1):
        if labels[i] not in keep:
            a = np.take(a, 0, axis=i)
            labels.pop(i)
    if 'Y' not in labels or 'X' not in labels:
        raise ValueError(f'Decoded array has no Y/X axes: axes={labels}, shape={a.shape}')
    if channel_label is None:
        yx = np.moveaxis(a, (labels.index('Y'), labels.index('X')), (0, 1))
        return (np.asarray(yx)[..., None], ['gray'])
    yxc = np.moveaxis(a, (labels.index('Y'), labels.index('X'), labels.index(channel_label)), (0, 1, 2))
    return (np.asarray(yxc), [f'channel_{i}' for i in range(yxc.shape[2])])

def _robust_u8(channel: np.ndarray, percentiles: tuple[float, float]) -> tuple[np.ndarray, float, float]:
    x = np.asarray(channel, np.float32)
    sample = x[::max(1, x.shape[0] // 512), ::max(1, x.shape[1] // 512)]
    sample = sample[np.isfinite(sample)]
    if not len(sample):
        return (np.zeros(x.shape, np.uint8), 0.0, 1.0)
    lo, hi = np.percentile(sample, percentiles)
    if hi <= lo:
        hi = lo + 1.0
    out = np.clip((x - lo) / (hi - lo) * 255.0, 0, 255).astype(np.uint8)
    return (out, float(lo), float(hi))

def _channel_structure_score(u8: np.ndarray) -> dict[str, float]:
    h, w = u8.shape
    scale = min(1.0, 512.0 / max(h, w))
    small = cv2.resize(u8, (max(32, round(w * scale)), max(32, round(h * scale))), interpolation=cv2.INTER_AREA) if scale < 1 else u8
    gx = cv2.Sobel(small, cv2.CV_32F, 1, 0, ksize=3)
    gy = cv2.Sobel(small, cv2.CV_32F, 0, 1, ksize=3)
    gradient = float(np.percentile(np.hypot(gx, gy), 90))
    coverage = float((small > 20).mean())
    hist = np.bincount(small.ravel(), minlength=256).astype(np.float64)
    p = hist / max(hist.sum(), 1.0)
    entropy = float(-(p[p > 0] * np.log2(p[p > 0])).sum())
    coverage_weight = math.exp(-((coverage - 0.25) / 0.35) ** 2)
    score = math.log1p(gradient) * (0.5 + entropy / 8.0) * (0.35 + coverage_weight)
    return {'score': float(score), 'gradient_p90': gradient, 'coverage': coverage, 'entropy': entropy}

def _array_to_he_rgb(arr: np.ndarray, axes: str) -> np.ndarray:
    yxc, _ = _reduce_to_yxc(arr, axes)
    if yxc.shape[2] == 1:
        yxc = np.repeat(yxc, 3, axis=2)
    elif yxc.shape[2] == 2:
        yxc = np.concatenate([yxc, yxc[..., :1]], axis=2)
    else:
        yxc = yxc[..., :3]
    if yxc.dtype == np.uint8:
        return np.ascontiguousarray(yxc)
    channels = [_robust_u8(yxc[..., i], (0.5, 99.5))[0] for i in range(3)]
    return np.ascontiguousarray(np.stack(channels, axis=2))

def _array_to_mif_structural(arr: np.ndarray, axes: str, cfg: Config) -> tuple[np.ndarray, pd.DataFrame, list[int]]:
    yxc, names = _reduce_to_yxc(arr, axes)
    normalized, rows = ([], [])
    for i in range(yxc.shape[2]):
        u8, lo, hi = _robust_u8(yxc[..., i], cfg.MIF_PERCENTILES)
        metrics = _channel_structure_score(u8)
        normalized.append(u8)
        rows.append({'channel_index': i, 'channel_name': names[i], 'normalization_low': lo, 'normalization_high': hi, **metrics})
    table = pd.DataFrame(rows).sort_values('score', ascending=False).reset_index(drop=True)
    if cfg.MIF_DAPI_CHANNEL is not None:
        idx = int(cfg.MIF_DAPI_CHANNEL)
        if idx < 0 or idx >= len(normalized):
            raise ValueError(f'MIF_DAPI_CHANNEL={idx}, but only {len(normalized)} channels are available')
        selected = [idx]
    else:
        selected = table.head(max(1, cfg.MIF_AUTO_TOP_CHANNELS))['channel_index'].astype(int).tolist()
    projection = np.max(np.stack([normalized[i] for i in selected], axis=0), axis=0)
    structural_rgb = np.repeat(projection[..., None], 3, axis=2)
    table['selected'] = table['channel_index'].isin(selected)
    return (np.ascontiguousarray(structural_rgb), table, selected)

def read_registration_image(path: Path, modality: str, cfg: Config=CFG) -> tuple[np.ndarray, dict[str, Any], Optional[pd.DataFrame]]:
    meta = inspect_qptiff(path)
    level_index = _choose_level(meta['levels'], cfg.PYRAMID_TARGET_LONG_EDGE)
    with tifffile.TiffFile(path) as tif:
        series = tif.series[meta['series_index']]
        level = list(getattr(series, 'levels', [series]))[level_index]
        arr = level.asarray()
        if modality == 'HE':
            rgb = _array_to_he_rgb(arr, level.axes)
            channel_table = None
        elif modality == 'mIF':
            rgb, channel_table, selected = _array_to_mif_structural(arr, level.axes, cfg)
            meta['mif_channel_count'] = int(len(channel_table))
            meta['mif_selected_channels'] = selected
        else:
            raise ValueError(modality)
    before = rgb.shape[:2]
    if max(rgb.shape[:2]) > cfg.PYRAMID_MAX_LONG_EDGE:
        scale = cfg.PYRAMID_MAX_LONG_EDGE / max(rgb.shape[:2])
        rgb = cv2.resize(rgb, (max(32, round(rgb.shape[1] * scale)), max(32, round(rgb.shape[0] * scale))), cv2.INTER_AREA)
    h, w = rgb.shape[:2]
    low_to_full = np.array([[meta['full_width'] / w, 0, 0], [0, meta['full_height'] / h, 0], [0, 0, 1]], np.float64)
    meta.update({'selected_level': level_index, 'selected_level_shape_before_resize': tuple((int(x) for x in before)), 'registration_height': h, 'registration_width': w, 'low_to_full': low_to_full, 'full_to_low': np.linalg.inv(low_to_full)})
    return (rgb, meta, channel_table)


## Slide inventory


In [ ]:
for label, directory in [('H&E', HE_DIR), ('mIF', MIF_DIR)]:
    if not directory.exists():
        raise FileNotFoundError(f'Missing {label} directory: {directory}')
he_files = sorted([p for p in HE_DIR.iterdir() if p.is_file() and p.suffix.lower() in CFG.EXTENSIONS], key=lambda p: p.name.lower())
mif_files = sorted([p for p in MIF_DIR.rglob('*') if p.is_file() and p.suffix.lower() in CFG.EXTENSIONS], key=lambda p: str(p).lower())
if not he_files:
    raise RuntimeError(f'No H&E QPTIFF found in {HE_DIR}')
if not mif_files:
    raise RuntimeError(f'No mIF QPTIFF found in {MIF_DIR}')

def extract_case_id(path: Path) -> str:
    match = re.match(CFG.CASE_ID_REGEX, path.stem, flags=re.IGNORECASE)
    if not match:
        raise ValueError(f'Invalid file name: {path.name}')
    return match.group('case_id')
inventory_rows = []
for modality, root, files in [('HE_fixed', HE_DIR, he_files), ('mIF_moving', MIF_DIR, mif_files)]:
    for path in tqdm(files, desc=f'Read {modality} metadata'):
        info = inspect_qptiff(path)
        level_index = _choose_level(info['levels'], CFG.PYRAMID_TARGET_LONG_EDGE)
        level = info['levels'][level_index]
        inventory_rows.append({'case_id': extract_case_id(path), 'modality': modality, 'file': path.name, 'relative_path': str(path.relative_to(root)), 'full_height': info['full_height'], 'full_width': info['full_width'], 'n_levels': len(info['levels']), 'selected_level': level_index, 'selected_height': level['height'], 'selected_width': level['width']})
inventory_df = pd.DataFrame(inventory_rows)
inventory_df.to_csv(OUTPUT_DIR / 'qptiff_inventory.csv', index=False, encoding='utf-8-sig')
display(inventory_df)


## Case pairing


In [ ]:
def unique_case_map(files: list[Path], modality: str) -> dict[str, Path]:
    grouped: dict[str, list[Path]] = {}
    for path in files:
        grouped.setdefault(extract_case_id(path), []).append(path)
    duplicates = {case_id: paths for case_id, paths in grouped.items() if len(paths) != 1}
    if duplicates:
        details = '\n'.join((f'{case_id}: {[str(x) for x in paths]}' for case_id, paths in duplicates.items()))
        raise RuntimeError(f'Duplicate {modality} cases:\n{details}')
    return {case_id: paths[0] for case_id, paths in grouped.items()}
he_by_case = unique_case_map(he_files, 'H&E')
mif_by_case = unique_case_map(mif_files, 'mIF')
he_ids, mif_ids = (set(he_by_case), set(mif_by_case))
only_he = sorted(he_ids - mif_ids)
only_mif = sorted(mif_ids - he_ids)
if only_he or only_mif:
    raise RuntimeError(f'H&E and mIF case sets do not match.\nH&E only: {only_he}\nmIF only: {only_mif}')
case_ids = sorted(he_ids & mif_ids)
if CFG.RUN_CASE_IDS:
    unknown = sorted(set(CFG.RUN_CASE_IDS) - set(case_ids))
    if unknown:
        raise ValueError(f'Unknown RUN_CASE_IDS: {unknown}')
    case_ids = [case_id for case_id in case_ids if case_id in set(CFG.RUN_CASE_IDS)]
pairs = [(he_by_case[case_id], mif_by_case[case_id]) for case_id in case_ids]
for fixed_path, moving_path in pairs:
    assert extract_case_id(fixed_path) == extract_case_id(moving_path), f'Cross-case pair: {moving_path.name} -> {fixed_path.name}'
pair_df = pd.DataFrame([{'case_id': extract_case_id(fixed_path), 'fixed_modality': 'H&E', 'fixed': str(fixed_path), 'moving_modality': 'mIF', 'moving': str(moving_path), 'verified_same_case': extract_case_id(fixed_path) == extract_case_id(moving_path)} for fixed_path, moving_path in pairs])
pair_df.to_csv(OUTPUT_DIR / 'registration_pairs.csv', index=False, encoding='utf-8-sig')
display(pair_df)
if not pair_df['verified_same_case'].all():
    raise RuntimeError('Pair audit failed')


## Structural views and model


In [ ]:
processor = AutoImageProcessor.from_pretrained(CFG.MODEL_ID)
matcher = AutoModelForKeypointMatching.from_pretrained(CFG.MODEL_ID).to(DEVICE).eval()

def clean_binary_mask(mask: np.ndarray) -> np.ndarray:
    mask8 = (np.asarray(mask) > 0).astype(np.uint8) * 255
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7, 7))
    mask8 = cv2.morphologyEx(mask8, cv2.MORPH_CLOSE, kernel, iterations=2)
    mask8 = cv2.morphologyEx(mask8, cv2.MORPH_OPEN, kernel, iterations=1)
    n, labels, stats, _ = cv2.connectedComponentsWithStats(mask8, connectivity=8)
    keep = np.zeros_like(mask8)
    min_area = max(32, int(mask8.size * 5e-05))
    for index in range(1, n):
        if stats[index, cv2.CC_STAT_AREA] >= min_area:
            keep[labels == index] = 255
    return keep.astype(bool)

def tissue_mask(rgb: np.ndarray, modality: str) -> np.ndarray:
    rgb = np.asarray(rgb, np.uint8)
    if modality == 'HE':
        hsv = cv2.cvtColor(rgb, cv2.COLOR_RGB2HSV)
        gray = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY)
        mask = (gray < 242) & ((hsv[..., 1] > 10) | (gray < 220))
    elif modality == 'mIF':
        signal = rgb.max(axis=2)
        otsu, _ = cv2.threshold(signal, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
        threshold = max(4.0, 0.5 * float(otsu))
        mask = signal > threshold
    else:
        raise ValueError(modality)
    return clean_binary_mask(mask)

def matching_view(rgb: np.ndarray, mask: np.ndarray, modality: str) -> np.ndarray:
    if modality == 'HE':
        signal = 255 - cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY)
    elif modality == 'mIF':
        signal = rgb.max(axis=2)
    else:
        raise ValueError(modality)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    enhanced = clahe.apply(signal.astype(np.uint8))
    view = 255 - enhanced
    view[~mask] = 255
    return np.repeat(view[..., None], 3, axis=2)

def letterbox_rgb(image: np.ndarray, canvas: int) -> tuple[np.ndarray, np.ndarray]:
    h, w = image.shape[:2]
    scale = min(canvas / w, canvas / h)
    new_w = max(1, min(canvas, int(round(w * scale))))
    new_h = max(1, min(canvas, int(round(h * scale))))
    interpolation = cv2.INTER_AREA if scale < 1 else cv2.INTER_CUBIC
    resized = cv2.resize(image, (new_w, new_h), interpolation=interpolation)
    x0, y0 = ((canvas - new_w) // 2, (canvas - new_h) // 2)
    out = np.full((canvas, canvas, 3), 255, np.uint8)
    out[y0:y0 + new_h, x0:x0 + new_w] = resized
    matrix = np.array([[new_w / w, 0.0, x0], [0.0, new_h / h, y0], [0.0, 0.0, 1.0]], dtype=np.float64)
    return (out, matrix)

def transform_points(matrix: np.ndarray, points: np.ndarray) -> np.ndarray:
    points = np.asarray(points, np.float64).reshape(-1, 2)
    hom = np.c_[points, np.ones(len(points))]
    warped = (np.asarray(matrix, np.float64) @ hom.T).T
    out = np.full((len(points), 2), np.nan, np.float64)
    valid = np.abs(warped[:, 2]) > 1e-12
    out[valid] = warped[valid, :2] / warped[valid, 2:3]
    return out

def orient_image(image: np.ndarray, orientation: str) -> tuple[np.ndarray, np.ndarray]:
    h, w = image.shape[:2]
    if orientation == 'identity':
        return (image.copy(), np.eye(3, dtype=np.float64))
    if orientation == 'rot90':
        matrix = np.array([[0, -1, h - 1], [1, 0, 0], [0, 0, 1]], np.float64)
        return (cv2.rotate(image, cv2.ROTATE_90_CLOCKWISE), matrix)
    if orientation == 'rot180':
        matrix = np.array([[-1, 0, w - 1], [0, -1, h - 1], [0, 0, 1]], np.float64)
        return (cv2.rotate(image, cv2.ROTATE_180), matrix)
    if orientation == 'rot270':
        matrix = np.array([[0, 1, 0], [-1, 0, w - 1], [0, 0, 1]], np.float64)
        return (cv2.rotate(image, cv2.ROTATE_90_COUNTERCLOCKWISE), matrix)
    if orientation == 'flip_h':
        matrix = np.array([[-1, 0, w - 1], [0, 1, 0], [0, 0, 1]], np.float64)
        return (cv2.flip(image, 1), matrix)
    if orientation == 'flip_v':
        matrix = np.array([[1, 0, 0], [0, -1, h - 1], [0, 0, 1]], np.float64)
        return (cv2.flip(image, 0), matrix)
    raise ValueError(f'Unknown orientation: {orientation}')

def points_inside_mask(mask: np.ndarray, points: np.ndarray) -> np.ndarray:
    xy = np.round(points).astype(np.int64)
    h, w = mask.shape
    valid = np.isfinite(points).all(axis=1) & (xy[:, 0] >= 0) & (xy[:, 0] < w) & (xy[:, 1] >= 0) & (xy[:, 1] < h)
    inside = np.zeros(len(points), dtype=bool)
    indices = np.where(valid)[0]
    inside[indices] = mask[xy[indices, 1], xy[indices, 0]]
    return inside

def normalized_pair_descriptor(fixed_pts: np.ndarray, moving_pts: np.ndarray, fixed_shape: tuple[int, int], moving_shape: tuple[int, int]) -> np.ndarray:
    hf, wf = fixed_shape
    hm, wm = moving_shape
    return np.column_stack([fixed_pts[:, 0] / max(wf - 1, 1) * CFG.MODEL_CANVAS, fixed_pts[:, 1] / max(hf - 1, 1) * CFG.MODEL_CANVAS, moving_pts[:, 0] / max(wm - 1, 1) * CFG.MODEL_CANVAS, moving_pts[:, 1] / max(hm - 1, 1) * CFG.MODEL_CANVAS]).astype(np.float32)

def mutual_consistency_filter(f_fixed: np.ndarray, f_moving: np.ndarray, f_scores: np.ndarray, r_fixed: np.ndarray, r_moving: np.ndarray, r_scores: np.ndarray, fixed_shape: tuple[int, int], moving_shape: tuple[int, int]) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    if not len(f_scores) or not len(r_scores):
        return (f_fixed[:0], f_moving[:0], f_scores[:0])
    desc_f = normalized_pair_descriptor(f_fixed, f_moving, fixed_shape, moving_shape)
    desc_r = normalized_pair_descriptor(r_fixed, r_moving, fixed_shape, moving_shape)
    matcher4d = cv2.BFMatcher(cv2.NORM_L2, crossCheck=True)
    mutual = matcher4d.match(desc_f, desc_r)
    accepted = [m for m in mutual if m.distance <= CFG.MUTUAL_TOLERANCE_CANVAS_PX]
    if not accepted:
        return (f_fixed[:0], f_moving[:0], f_scores[:0])
    q = np.asarray([m.queryIdx for m in accepted], dtype=np.int64)
    t = np.asarray([m.trainIdx for m in accepted], dtype=np.int64)
    scores = np.sqrt(np.clip(f_scores[q], 0, 1) * np.clip(r_scores[t], 0, 1))
    return (f_fixed[q], f_moving[q], scores)

def match_orientation(fixed_view: np.ndarray, moving_view: np.ndarray, fixed_mask: np.ndarray, moving_mask: np.ndarray, orientation: str) -> dict[str, Any]:
    oriented_moving, moving_to_oriented = orient_image(moving_view, orientation)
    oriented_mask, _ = orient_image(moving_mask.astype(np.uint8), orientation)
    oriented_mask = oriented_mask.astype(bool)
    forward = infer_correspondences(fixed_view, oriented_moving)
    f_fixed = forward['pts0']
    f_oriented = forward['pts1']
    f_moving = transform_points(np.linalg.inv(moving_to_oriented), f_oriented)
    valid_f = points_inside_mask(fixed_mask, f_fixed) & points_inside_mask(moving_mask, f_moving)
    f_fixed, f_moving, f_scores = (f_fixed[valid_f], f_moving[valid_f], forward['scores'][valid_f])
    reverse_count = 0
    mutual_used = False
    n_after_mutual = 0
    if CFG.USE_BIDIRECTIONAL_MATCHING:
        reverse = infer_correspondences(oriented_moving, fixed_view)
        reverse_count = reverse['n_model_candidates']
        r_oriented = reverse['pts0']
        r_fixed = reverse['pts1']
        r_moving = transform_points(np.linalg.inv(moving_to_oriented), r_oriented)
        valid_r = points_inside_mask(fixed_mask, r_fixed) & points_inside_mask(moving_mask, r_moving)
        r_fixed, r_moving, r_scores = (r_fixed[valid_r], r_moving[valid_r], reverse['scores'][valid_r])
        m_fixed, m_moving, m_scores = mutual_consistency_filter(f_fixed, f_moving, f_scores, r_fixed, r_moving, r_scores, fixed_view.shape[:2], moving_view.shape[:2])
        n_after_mutual = int(len(m_scores))
        if n_after_mutual >= CFG.MIN_MUTUAL_MATCHES:
            f_fixed, f_moving, f_scores = (m_fixed, m_moving, m_scores)
            mutual_used = True
    return {'orientation': orientation, 'fixed_pts': f_fixed, 'moving_pts': f_moving, 'scores': f_scores, 'n_model_candidates': forward['n_model_candidates'], 'n_reverse_model_candidates': reverse_count, 'n_after_tissue_filter': int(valid_f.sum()), 'n_after_mutual_filter': n_after_mutual, 'mutual_used': mutual_used, 'moving_to_oriented': moving_to_oriented}


## Robust global matching


In [ ]:
def spatial_coverage(points: np.ndarray, image_shape: tuple[int, int]) -> float:
    if len(points) < 3:
        return 0.0
    hull = cv2.convexHull(np.asarray(points, np.float32))
    h, w = image_shape
    return float(cv2.contourArea(hull)) / max(float(h * w), 1.0)

def model_metrics(matrix: Optional[np.ndarray], inliers: Optional[np.ndarray], moving_pts: np.ndarray, fixed_pts: np.ndarray, moving_shape: tuple[int, int], fixed_shape: tuple[int, int]) -> dict[str, Any]:
    if matrix is None or inliers is None:
        return {'valid': False, 'n_inliers': 0, 'inlier_ratio': 0.0}
    mask = np.asarray(inliers).reshape(-1).astype(bool)
    projected = transform_points(matrix, moving_pts)
    errors = np.linalg.norm(projected - fixed_pts, axis=1)
    accepted = errors[mask]
    return {'valid': True, 'n_candidate_matches': int(len(mask)), 'n_inliers': int(mask.sum()), 'n_outliers': int(len(mask) - mask.sum()), 'inlier_ratio': float(mask.mean()) if len(mask) else 0.0, 'rmse_px': float(np.sqrt(np.mean(accepted ** 2))) if len(accepted) else None, 'median_error_px': float(np.median(accepted)) if len(accepted) else None, 'p95_error_px': float(np.percentile(accepted, 95)) if len(accepted) else None, 'moving_coverage': spatial_coverage(moving_pts[mask], moving_shape), 'fixed_coverage': spatial_coverage(fixed_pts[mask], fixed_shape)}

def geometry_diagnostics(matrix: np.ndarray, fixed_shape: tuple[int, int]) -> dict[str, Any]:
    linear = np.asarray(matrix, np.float64)[:2, :2]
    singular = np.linalg.svd(linear, compute_uv=False)
    scale_max, scale_min = (float(singular.max()), float(singular.min()))
    determinant = float(np.linalg.det(linear))
    anisotropy = scale_max / max(scale_min, 1e-12)
    rotation_deg = float(np.degrees(np.arctan2(linear[1, 0], linear[0, 0])))
    translation = np.asarray(matrix, np.float64)[:2, 2]
    h, w = fixed_shape
    translation_fraction = float(np.linalg.norm(translation) / max(math.hypot(w, h), 1.0))
    plausible = scale_min >= CFG.MIN_SCALE and scale_max <= CFG.MAX_SCALE and (anisotropy <= CFG.MAX_ANISOTROPY) and (translation_fraction <= CFG.MAX_TRANSLATION_FRACTION) and (abs(determinant) > 1e-08) and (CFG.ALLOW_REFLECTION or determinant > 0)
    return {'scale_min': scale_min, 'scale_max': scale_max, 'anisotropy': anisotropy, 'determinant': determinant, 'reflection': determinant < 0, 'rotation_deg': rotation_deg, 'translation_fraction': translation_fraction, 'plausible': bool(plausible)}

def estimate_one_model(model_name: str, moving_pts: np.ndarray, fixed_pts: np.ndarray) -> tuple[Optional[np.ndarray], Optional[np.ndarray]]:
    if len(moving_pts) < 3:
        return (None, None)
    if model_name == 'similarity':
        affine2, mask = cv2.estimateAffinePartial2D(moving_pts.astype(np.float32), fixed_pts.astype(np.float32), method=cv2.RANSAC, ransacReprojThreshold=CFG.RANSAC_THRESHOLD_PX, maxIters=CFG.RANSAC_MAX_ITERS, confidence=CFG.RANSAC_CONFIDENCE, refineIters=25)
    elif model_name == 'affine':
        method = getattr(cv2, 'USAC_MAGSAC', cv2.RANSAC)
        try:
            affine2, mask = cv2.estimateAffine2D(moving_pts.astype(np.float32), fixed_pts.astype(np.float32), method=method, ransacReprojThreshold=CFG.RANSAC_THRESHOLD_PX, maxIters=CFG.RANSAC_MAX_ITERS, confidence=CFG.RANSAC_CONFIDENCE, refineIters=25)
        except cv2.error:
            affine2, mask = cv2.estimateAffine2D(moving_pts.astype(np.float32), fixed_pts.astype(np.float32), method=cv2.RANSAC, ransacReprojThreshold=CFG.RANSAC_THRESHOLD_PX, maxIters=CFG.RANSAC_MAX_ITERS, confidence=CFG.RANSAC_CONFIDENCE, refineIters=25)
    else:
        raise ValueError(model_name)
    if affine2 is None:
        return (None, None)
    matrix = np.vstack([affine2, [0.0, 0.0, 1.0]]).astype(np.float64)
    return (matrix, None if mask is None else mask.reshape(-1).astype(bool))

def grid_balance_indices(fixed_pts: np.ndarray, moving_pts: np.ndarray, scores: np.ndarray, fixed_shape: tuple[int, int], moving_shape: tuple[int, int]) -> np.ndarray:
    if not len(scores):
        return np.empty(0, dtype=np.int64)
    hf, wf = fixed_shape
    hm, wm = moving_shape
    gx_f = np.clip((fixed_pts[:, 0] / max(wf, 1) * CFG.GRID_SIZE).astype(int), 0, CFG.GRID_SIZE - 1)
    gy_f = np.clip((fixed_pts[:, 1] / max(hf, 1) * CFG.GRID_SIZE).astype(int), 0, CFG.GRID_SIZE - 1)
    gx_m = np.clip((moving_pts[:, 0] / max(wm, 1) * CFG.GRID_SIZE).astype(int), 0, CFG.GRID_SIZE - 1)
    gy_m = np.clip((moving_pts[:, 1] / max(hm, 1) * CFG.GRID_SIZE).astype(int), 0, CFG.GRID_SIZE - 1)
    counts_f: dict[tuple[int, int], int] = {}
    counts_m: dict[tuple[int, int], int] = {}
    selected = []
    for i in np.argsort(scores)[::-1]:
        cf, cm = ((int(gx_f[i]), int(gy_f[i])), (int(gx_m[i]), int(gy_m[i])))
        if counts_f.get(cf, 0) >= CFG.MAX_MATCHES_PER_GRID_CELL:
            continue
        if counts_m.get(cm, 0) >= CFG.MAX_MATCHES_PER_GRID_CELL:
            continue
        selected.append(int(i))
        counts_f[cf] = counts_f.get(cf, 0) + 1
        counts_m[cm] = counts_m.get(cm, 0) + 1
        if len(selected) >= CFG.MAX_RANSAC_CANDIDATES:
            break
    return np.asarray(selected, dtype=np.int64)

def candidate_quality(candidate: dict[str, Any]) -> float:
    metrics, geometry = (candidate['metrics'], candidate['geometry'])
    if not metrics.get('valid') or not geometry.get('plausible'):
        return -1.0
    n = metrics['n_inliers']
    ratio = metrics['inlier_ratio']
    coverage = min(metrics['fixed_coverage'], metrics['moving_coverage'])
    median = metrics.get('median_error_px')
    median = CFG.RANSAC_THRESHOLD_PX if median is None else median
    score = math.log1p(n) * max(ratio, 1e-06) ** 3 * math.sqrt(0.01 + coverage)
    score /= 1.0 + median / max(CFG.RANSAC_THRESHOLD_PX, 1.0)
    if not candidate.get('mutual_used', False):
        score *= 0.2
    if candidate['model'] == 'affine':
        score *= 0.85
    return float(score)

def evaluate_match_result(match_result: dict[str, Any], fixed_shape: tuple[int, int], moving_shape: tuple[int, int]) -> list[dict[str, Any]]:
    candidates = []
    all_scores = match_result['scores']
    for threshold in CFG.SCORE_THRESHOLDS:
        keep = all_scores >= threshold
        fixed_raw = match_result['fixed_pts'][keep]
        moving_raw = match_result['moving_pts'][keep]
        scores_raw = all_scores[keep]
        if len(scores_raw) < 3:
            continue
        balanced = grid_balance_indices(fixed_raw, moving_raw, scores_raw, fixed_shape, moving_shape)
        fixed_pts, moving_pts, scores = (fixed_raw[balanced], moving_raw[balanced], scores_raw[balanced])
        if len(scores) < 3:
            continue
        model_names = ['similarity'] + (['affine'] if CFG.ALLOW_AFFINE else [])
        for model_name in model_names:
            matrix, inliers = estimate_one_model(model_name, moving_pts, fixed_pts)
            metrics = model_metrics(matrix, inliers, moving_pts, fixed_pts, moving_shape, fixed_shape)
            geometry = geometry_diagnostics(matrix, fixed_shape) if matrix is not None else {'plausible': False}
            candidate = {'orientation': match_result['orientation'], 'score_threshold': float(threshold), 'model': model_name, 'matrix': matrix, 'inliers': inliers, 'fixed_pts': fixed_pts, 'moving_pts': moving_pts, 'scores': scores, 'metrics': metrics, 'geometry': geometry, 'n_model_candidates': match_result['n_model_candidates'], 'n_reverse_model_candidates': match_result['n_reverse_model_candidates'], 'n_after_tissue_filter': match_result['n_after_tissue_filter'], 'n_after_mutual_filter': match_result['n_after_mutual_filter'], 'n_before_grid_filter': int(len(scores_raw)), 'n_after_grid_filter': int(len(scores)), 'mutual_used': match_result['mutual_used']}
            candidate['quality_score'] = candidate_quality(candidate)
            candidates.append(candidate)
    return candidates

def choose_best(candidates: list[dict[str, Any]]) -> Optional[dict[str, Any]]:
    valid = [c for c in candidates if c['quality_score'] >= 0 and c['geometry'].get('plausible', False) and (c['metrics'].get('n_inliers', 0) >= 3)]
    if not valid:
        return None
    usable = []
    for c in valid:
        m = c['metrics']
        coverage = min(m.get('fixed_coverage', 0.0), m.get('moving_coverage', 0.0))
        if m.get('n_inliers', 0) >= CFG.HARD_MIN_INLIERS and m.get('inlier_ratio', 0.0) >= CFG.FAIL_MIN_INLIER_RATIO and (coverage >= CFG.FAIL_MIN_SPATIAL_COVERAGE):
            usable.append(c)
    pool = usable if usable else valid
    strict = []
    for c in pool:
        m = c['metrics']
        coverage = min(m.get('fixed_coverage', 0.0), m.get('moving_coverage', 0.0))
        if c.get('mutual_used', False) and m.get('n_inliers', 0) >= CFG.QC_MIN_INLIERS and (m.get('inlier_ratio', 0.0) >= CFG.QC_MIN_INLIER_RATIO) and (coverage >= CFG.QC_MIN_SPATIAL_COVERAGE):
            strict.append(c)
    return max(strict if strict else pool, key=lambda c: c['quality_score'])

def fast_accept(candidate: Optional[dict[str, Any]]) -> bool:
    if candidate is None:
        return False
    m = candidate['metrics']
    coverage = min(m.get('fixed_coverage', 0.0), m.get('moving_coverage', 0.0))
    return candidate.get('mutual_used', False) and candidate['geometry'].get('plausible', False) and (m.get('n_inliers', 0) >= CFG.FAST_ACCEPT_MIN_INLIERS) and (m.get('inlier_ratio', 0.0) >= CFG.FAST_ACCEPT_MIN_RATIO) and (coverage >= CFG.FAST_ACCEPT_MIN_COVERAGE)


## QC exports


In [ ]:
def safe_slug(text: str) -> str:
    return re.sub('[^0-9A-Za-z._-]+', '_', text).strip('._') or 'case'

def json_ready(obj: Any) -> Any:
    if isinstance(obj, Path):
        return str(obj)
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        return float(obj)
    if isinstance(obj, (np.bool_,)):
        return bool(obj)
    if isinstance(obj, dict):
        return {str(k): json_ready(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [json_ready(x) for x in obj]
    return obj

def save_rgb(path: Path, rgb: np.ndarray) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    if not cv2.imwrite(str(path), cv2.cvtColor(np.asarray(rgb, np.uint8), cv2.COLOR_RGB2BGR)):
        raise IOError(f'Cannot save: {path}')

def warp_rgb(image: np.ndarray, matrix: np.ndarray, fixed_shape: tuple[int, int]) -> np.ndarray:
    h, w = fixed_shape
    return cv2.warpPerspective(image, matrix.astype(np.float64), (w, h), flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_CONSTANT, borderValue=(0, 0, 0))

def warp_mask(mask: np.ndarray, matrix: np.ndarray, fixed_shape: tuple[int, int]) -> np.ndarray:
    h, w = fixed_shape
    return cv2.warpPerspective(mask.astype(np.uint8), matrix.astype(np.float64), (w, h), flags=cv2.INTER_NEAREST, borderMode=cv2.BORDER_CONSTANT, borderValue=0).astype(bool)

def tissue_overlap_metrics(fixed_mask: np.ndarray, moving_mask: np.ndarray) -> dict[str, float]:
    intersection = int((fixed_mask & moving_mask).sum())
    a, b = (int(fixed_mask.sum()), int(moving_mask.sum()))
    return {'tissue_dice': 2.0 * intersection / max(a + b, 1), 'fixed_tissue_covered': intersection / max(a, 1), 'moving_tissue_covered': intersection / max(b, 1)}

def make_checkerboard(a: np.ndarray, b: np.ndarray, block: int=96) -> np.ndarray:
    yy, xx = np.indices(a.shape[:2])
    return np.where(((xx // block + yy // block) % 2 == 0)[..., None], a, b).astype(np.uint8)

def make_tissue_contour_qc(fixed_rgb, fixed_mask, warped_mask) -> np.ndarray:
    canvas = fixed_rgb.copy()
    fc, _ = cv2.findContours(fixed_mask.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    mc, _ = cv2.findContours(warped_mask.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    cv2.drawContours(canvas, fc, -1, (20, 210, 60), 3)
    cv2.drawContours(canvas, mc, -1, (235, 40, 210), 3)
    return canvas

def full_resolution_matrix(low_matrix, fixed_low_to_full, moving_low_to_full) -> np.ndarray:
    out = fixed_low_to_full @ low_matrix @ np.linalg.inv(moving_low_to_full)
    return out / out[2, 2] if abs(out[2, 2]) > 1e-12 else out

def mask_pca_initial(fixed_mask: np.ndarray, moving_mask: np.ndarray) -> tuple[np.ndarray, float]:

    def stats(mask):
        yx = np.column_stack(np.nonzero(mask))
        if len(yx) < 20:
            raise RuntimeError('Insufficient tissue pixels for PCA initialization')
        xy = yx[:, ::-1].astype(np.float64)
        if len(xy) > 100000:
            xy = xy[::math.ceil(len(xy) / 100000)]
        center = xy.mean(axis=0)
        cov = np.cov((xy - center).T)
        values, vectors = np.linalg.eigh(cov)
        axis = vectors[:, np.argmax(values)]
        angle = math.atan2(axis[1], axis[0])
        return (center, angle, float(mask.sum()))
    cf, af, area_f = stats(fixed_mask)
    cm, am, area_m = stats(moving_mask)
    scale = float(np.clip(math.sqrt(area_f / max(area_m, 1.0)), CFG.MIN_SCALE, CFG.MAX_SCALE))
    candidates = []
    for extra in (0.0, math.pi):
        angle = af - am + extra
        c, s = (math.cos(angle) * scale, math.sin(angle) * scale)
        linear = np.array([[c, -s], [s, c]], np.float64)
        translation = cf - linear @ cm
        matrix = np.array([[linear[0, 0], linear[0, 1], translation[0]], [linear[1, 0], linear[1, 1], translation[1]], [0, 0, 1]], np.float64)
        warped = warp_mask(moving_mask, matrix, fixed_mask.shape)
        dice = tissue_overlap_metrics(fixed_mask, warped)['tissue_dice']
        candidates.append((dice, matrix))
    return (max(candidates, key=lambda x: x[0])[1], float(max((x[0] for x in candidates))))

def load_v3_global_matrix(case_id: str) -> tuple[Optional[np.ndarray], Optional[dict[str, Any]], str]:
    pair_dir = V3_RESULTS_DIR / 'pairs' / f'{safe_slug(case_id)}__mIF_to_HE'
    for name in ('transform_registration.npy', 'transform_candidate_registration.npy'):
        path = pair_dir / name
        if path.exists():
            report = None
            metrics_path = pair_dir / 'metrics.json'
            if metrics_path.exists():
                try:
                    report = json.loads(metrics_path.read_text(encoding='utf-8'))
                except Exception:
                    pass
            return (np.load(path).astype(np.float64), report, f'v3:{name}')
    return (None, None, 'not_available')

def estimate_global_initial(case_id: str, fixed_view: np.ndarray, moving_view: np.ndarray, fixed_mask: np.ndarray, moving_mask: np.ndarray, pair_dir: Path) -> tuple[np.ndarray, dict[str, Any]]:
    if CFG.REUSE_V3_GLOBAL:
        matrix, old_report, source = load_v3_global_matrix(case_id)
        if matrix is not None:
            return (matrix, {'source': source, 'v3_report': old_report})
    all_candidates = []
    orientations = ['identity']
    result = match_orientation(fixed_view, moving_view, fixed_mask, moving_mask, 'identity')
    all_candidates.extend(evaluate_match_result(result, fixed_view.shape[:2], moving_view.shape[:2]))
    best = choose_best(all_candidates)
    if CFG.ENABLE_ORIENTATION_RESCUE and (not fast_accept(best)):
        for orientation in CFG.RESCUE_ORIENTATIONS:
            orientations.append(orientation)
            result = match_orientation(fixed_view, moving_view, fixed_mask, moving_mask, orientation)
            all_candidates.extend(evaluate_match_result(result, fixed_view.shape[:2], moving_view.shape[:2]))
        best = choose_best(all_candidates)
    rows = []
    for c in all_candidates:
        m, g = (c['metrics'], c['geometry'])
        rows.append({'orientation': c['orientation'], 'threshold': c['score_threshold'], 'model': c['model'], 'quality_score': c['quality_score'], 'n_candidates': m.get('n_candidate_matches', 0), 'n_inliers': m.get('n_inliers', 0), 'inlier_ratio': m.get('inlier_ratio', 0), 'fixed_coverage': m.get('fixed_coverage', 0), 'moving_coverage': m.get('moving_coverage', 0), 'plausible': g.get('plausible', False)})
    pd.DataFrame(rows).to_csv(pair_dir / 'global_candidate_search.csv', index=False, encoding='utf-8-sig')
    if best is not None and best['metrics'].get('n_inliers', 0) >= 3:
        return (best['matrix'], {'source': 'fresh_eloftr', 'orientations': orientations, 'selected_model': best['model'], 'selected_threshold': best['score_threshold'], 'metrics': best['metrics'], 'geometry': best['geometry']})
    matrix, dice = mask_pca_initial(fixed_mask, moving_mask)
    return (matrix, {'source': 'tissue_mask_pca', 'initial_dice': dice})

def tile_origins(length: int, tile: int, stride: int) -> list[int]:
    if length <= tile:
        return [0]
    values = list(range(0, length - tile + 1, stride))
    if values[-1] != length - tile:
        values.append(length - tile)
    return values

def choose_overlap_tiles(overlap: np.ndarray) -> list[tuple[int, int, int]]:
    h, w = overlap.shape
    candidates = []
    for y in tile_origins(h, CFG.TILE_SIZE, CFG.TILE_STRIDE):
        for x in tile_origins(w, CFG.TILE_SIZE, CFG.TILE_STRIDE):
            patch = overlap[y:y + CFG.TILE_SIZE, x:x + CFG.TILE_SIZE]
            fraction = float(patch.mean())
            if fraction >= CFG.TILE_MIN_OVERLAP_FRACTION:
                candidates.append((round(fraction * patch.size), x, y))
    candidates.sort(reverse=True)
    selected = []
    for item in candidates:
        _, x, y = item
        if all((math.hypot(x - sx, y - sy) >= CFG.TILE_STRIDE * 0.65 for _, sx, sy in selected)):
            selected.append(item)
        if len(selected) >= CFG.MAX_TILES:
            break
    if len(selected) < min(CFG.MAX_TILES, len(candidates)):
        existing = {(x, y) for _, x, y in selected}
        selected.extend([c for c in candidates if (c[1], c[2]) not in existing][:CFG.MAX_TILES - len(selected)])
    return selected

def deduplicate_controls(fixed_pts, moving_pts, scores):
    order = np.argsort(scores)[::-1]
    seen_f, seen_m, keep = (set(), set(), [])
    b = max(CFG.CONTROL_DEDUP_BIN_PX, 1.0)
    for i in order:
        kf = tuple(np.floor(fixed_pts[i] / b).astype(int))
        km = tuple(np.floor(moving_pts[i] / b).astype(int))
        if kf in seen_f or km in seen_m:
            continue
        seen_f.add(kf)
        seen_m.add(km)
        keep.append(int(i))
        if len(keep) >= CFG.MAX_CONTROL_POINTS:
            break
    keep = np.asarray(keep, np.int64)
    return (fixed_pts[keep], moving_pts[keep], scores[keep])

def refine_affine(moving_pts, fixed_pts, global_matrix):
    if len(fixed_pts) < 3:
        return (global_matrix.copy(), np.zeros(len(fixed_pts), bool), {'accepted': False, 'reason': 'controls<3'})
    method = getattr(cv2, 'USAC_MAGSAC', cv2.RANSAC)
    try:
        affine2, mask = cv2.estimateAffine2D(moving_pts.astype(np.float32), fixed_pts.astype(np.float32), method=method, ransacReprojThreshold=CFG.REFINE_RANSAC_THRESHOLD_PX, maxIters=30000, confidence=0.999, refineIters=30)
    except cv2.error:
        affine2, mask = cv2.estimateAffine2D(moving_pts.astype(np.float32), fixed_pts.astype(np.float32), method=cv2.RANSAC, ransacReprojThreshold=CFG.REFINE_RANSAC_THRESHOLD_PX, maxIters=30000, confidence=0.999, refineIters=30)
    if affine2 is None or mask is None:
        return (global_matrix.copy(), np.zeros(len(fixed_pts), bool), {'accepted': False, 'reason': 'estimation_failed'})
    matrix = np.vstack([affine2, [0, 0, 1]]).astype(np.float64)
    inliers = mask.reshape(-1).astype(bool)
    h, w = (fixed_pts[:, 1].max(initial=1), fixed_pts[:, 0].max(initial=1))
    probes = np.array([[0, 0], [w, 0], [0, h], [w, h], [w / 2, h / 2]], np.float64)
    update = np.linalg.norm(transform_points(matrix, probes) - transform_points(global_matrix, probes), axis=1)
    accepted = int(inliers.sum()) >= CFG.HARD_MIN_INLIERS and float(np.median(update)) <= CFG.MAX_AFFINE_UPDATE_PX
    return (matrix if accepted else global_matrix.copy(), inliers, {'accepted': bool(accepted), 'n_inliers': int(inliers.sum()), 'inlier_ratio': float(inliers.mean()), 'median_global_update_px': float(np.median(update))})

def spatial_control_coverage(points: np.ndarray, shape: tuple[int, int]) -> float:
    if len(points) < 3:
        return 0.0
    hull = cv2.convexHull(points.astype(np.float32))
    return float(cv2.contourArea(hull)) / max(float(shape[0] * shape[1]), 1.0)

def fit_tps_package(moving_pts, fixed_pts, affine_matrix, fixed_meta, moving_meta):
    base = transform_points(affine_matrix, moving_pts)
    delta = fixed_pts - base
    enabled = len(base) >= CFG.TPS_MIN_CONTROLS
    result = {'enabled': enabled, 'base_fixed_registration': base, 'delta_fixed_registration': delta, 'moving_control_registration': moving_pts, 'fixed_control_registration': fixed_pts, 'affine_registration': affine_matrix, 'fixed_low_to_full': fixed_meta['low_to_full'], 'moving_full_to_low': moving_meta['full_to_low'], 'smoothing': CFG.TPS_SMOOTHING, 'neighbors': min(CFG.TPS_NEIGHBORS, max(len(base), 1))}
    if not enabled:
        result.update({'training_median_px': None, 'training_p95_px': None, 'max_displacement_px': None})
        return (result, None)
    neighbors = min(CFG.TPS_NEIGHBORS, len(base))
    try:
        rbf = RBFInterpolator(base, delta, kernel='thin_plate_spline', smoothing=CFG.TPS_SMOOTHING, neighbors=neighbors if neighbors < len(base) else None)
    except Exception as exc:
        result['enabled'] = False
        result.update({'training_median_px': None, 'training_p95_px': None, 'max_displacement_px': None, 'fit_error': f'{type(exc).__name__}: {exc}'})
        return (result, None)
    predicted_delta = rbf(base)
    norm = np.linalg.norm(predicted_delta, axis=1)
    predicted_delta *= np.minimum(1.0, CFG.TPS_MAX_DISPLACEMENT_PX / np.maximum(norm, 1e-06))[:, None]
    error = np.linalg.norm(base + predicted_delta - fixed_pts, axis=1)
    result.update({'training_median_px': float(np.median(error)), 'training_p95_px': float(np.percentile(error, 95)), 'max_displacement_px': float(norm.max(initial=0.0))})
    return (result, rbf)

def save_transform_package(path: Path, package: dict[str, Any]) -> None:
    np.savez_compressed(path, enabled=np.array([bool(package['enabled'])], np.uint8), base_fixed_registration=np.asarray(package['base_fixed_registration'], np.float64), delta_fixed_registration=np.asarray(package['delta_fixed_registration'], np.float64), moving_control_registration=np.asarray(package['moving_control_registration'], np.float64), fixed_control_registration=np.asarray(package['fixed_control_registration'], np.float64), affine_registration=np.asarray(package['affine_registration'], np.float64), fixed_low_to_full=np.asarray(package['fixed_low_to_full'], np.float64), moving_full_to_low=np.asarray(package['moving_full_to_low'], np.float64), smoothing=np.array([package['smoothing']], np.float64), neighbors=np.array([package['neighbors']], np.int32), max_displacement=np.array([CFG.TPS_MAX_DISPLACEMENT_PX], np.float64))

def evaluate_tps_displacement(rbf, points):
    delta = rbf(points)
    norm = np.linalg.norm(delta, axis=1)
    return delta * np.minimum(1.0, CFG.TPS_MAX_DISPLACEMENT_PX / np.maximum(norm, 1e-06))[:, None]

def warp_with_affine_tps(image, affine_matrix, rbf, fixed_shape, is_mask=False):
    if rbf is None:
        return warp_mask(image, affine_matrix, fixed_shape) if is_mask else warp_rgb(image, affine_matrix, fixed_shape)
    h, w = fixed_shape
    nx = max(2, math.ceil((w - 1) / CFG.FIELD_GRID_SPACING) + 1)
    ny = max(2, math.ceil((h - 1) / CFG.FIELD_GRID_SPACING) + 1)
    xs, ys = (np.linspace(0, w - 1, nx), np.linspace(0, h - 1, ny))
    xx, yy = np.meshgrid(xs, ys)
    target = np.column_stack([xx.ravel(), yy.ravel()])
    base = target.copy()
    for _ in range(3):
        base = target - evaluate_tps_displacement(rbf, base)
    source = transform_points(np.linalg.inv(affine_matrix), base).reshape(ny, nx, 2)
    map_x = cv2.resize(source[..., 0].astype(np.float32), (w, h), interpolation=cv2.INTER_CUBIC)
    map_y = cv2.resize(source[..., 1].astype(np.float32), (w, h), interpolation=cv2.INTER_CUBIC)
    interpolation = cv2.INTER_NEAREST if is_mask else cv2.INTER_LINEAR
    border = 0 if is_mask else (0, 0, 0)
    warped = cv2.remap(image.astype(np.uint8), map_x, map_y, interpolation, borderMode=cv2.BORDER_CONSTANT, borderValue=border)
    return warped.astype(bool) if is_mask else warped

def tps_fold_diagnostics(rbf, shape):
    if rbf is None:
        return {'fold_fraction': 0.0, 'min_jacobian': 1.0}
    h, w = shape
    xs, ys = (np.linspace(0, w - 1, 48), np.linspace(0, h - 1, 48))
    xx, yy = np.meshgrid(xs, ys)
    points = np.column_stack([xx.ravel(), yy.ravel()])
    delta = evaluate_tps_displacement(rbf, points).reshape(48, 48, 2)
    ddx_dx = np.gradient(delta[..., 0], xs, axis=1)
    ddx_dy = np.gradient(delta[..., 0], ys, axis=0)
    ddy_dx = np.gradient(delta[..., 1], xs, axis=1)
    ddy_dy = np.gradient(delta[..., 1], ys, axis=0)
    jac = (1 + ddx_dx) * (1 + ddy_dy) - ddx_dy * ddy_dx
    return {'fold_fraction': float((jac <= 0).mean()), 'min_jacobian': float(jac.min())}

def draw_controls_qc(fixed_rgb, fixed_pts, predicted_pts, max_points=300):
    canvas = fixed_rgb.copy()
    if not len(fixed_pts):
        return canvas
    order = np.linspace(0, len(fixed_pts) - 1, min(len(fixed_pts), max_points)).astype(int)
    for i in order:
        a = tuple(np.round(predicted_pts[i]).astype(int))
        b = tuple(np.round(fixed_pts[i]).astype(int))
        cv2.line(canvas, a, b, (230, 70, 70), 1, cv2.LINE_AA)
        cv2.circle(canvas, b, 3, (30, 210, 70), -1, cv2.LINE_AA)
    return canvas
_MODEL_CALL_COUNT = 0
_MODEL_RELOAD_TOTAL = 0
_EMPTY_RELOADS_THIS_CASE = 0

def reload_eloftr(reason: str='manual') -> None:
    global matcher, _MODEL_CALL_COUNT, _MODEL_RELOAD_TOTAL
    old = globals().get('matcher')
    if old is not None:
        try:
            old.to('cpu')
        except Exception:
            pass
        del old
    if 'matcher' in globals():
        del globals()['matcher']
    gc.collect()
    if DEVICE.type == 'cuda':
        torch.cuda.empty_cache()
        torch.cuda.synchronize()
    matcher = AutoModelForKeypointMatching.from_pretrained(CFG.MODEL_ID, local_files_only=True).to(DEVICE).eval()
    _MODEL_CALL_COUNT = 0
    _MODEL_RELOAD_TOTAL += 1

def prepare_matcher_for_case(case_id: str) -> None:
    global _EMPTY_RELOADS_THIS_CASE
    _EMPTY_RELOADS_THIS_CASE = 0
    if CFG.RELOAD_MODEL_EVERY_CASE:
        reload_eloftr(f'start_case:{case_id}')

def infer_correspondences(image0: np.ndarray, image1: np.ndarray, *, canvas: Optional[int]=None, threshold: Optional[float]=None) -> dict[str, Any]:
    global _MODEL_CALL_COUNT
    use_canvas = int(canvas or CFG.MODEL_CANVAS)
    use_threshold = float(CFG.MATCH_MIN_SCORE if threshold is None else threshold)
    if CFG.RELOAD_MODEL_EVERY_N_CALLS > 0 and _MODEL_CALL_COUNT >= CFG.RELOAD_MODEL_EVERY_N_CALLS:
        reload_eloftr(f'periodic_after_{_MODEL_CALL_COUNT}_calls')
    canvas0, to_canvas0 = letterbox_rgb(image0, use_canvas)
    canvas1, to_canvas1 = letterbox_rgb(image1, use_canvas)
    inputs = outputs = result = None
    try:
        inputs = processor([Image.fromarray(canvas0), Image.fromarray(canvas1)], do_resize=False, return_tensors='pt')
        inputs = {k: v.to(DEVICE, non_blocking=True) if torch.is_tensor(v) else v for k, v in inputs.items()}
        amp_enabled = CFG.USE_MIXED_PRECISION and DEVICE.type == 'cuda'
        amp_context = torch.autocast('cuda', dtype=torch.float16) if amp_enabled else nullcontext()
        with torch.inference_mode(), amp_context:
            outputs = matcher(**inputs)
        result = processor.post_process_keypoint_matching(outputs, target_sizes=[[(use_canvas, use_canvas), (use_canvas, use_canvas)]], threshold=use_threshold)[0]
        pts0_canvas = result['keypoints0'].detach().float().cpu().numpy().astype(np.float64)
        pts1_canvas = result['keypoints1'].detach().float().cpu().numpy().astype(np.float64)
        scores = result['matching_scores'].detach().float().cpu().numpy().astype(np.float64)
        answer = {'pts0': transform_points(np.linalg.inv(to_canvas0), pts0_canvas), 'pts1': transform_points(np.linalg.inv(to_canvas1), pts1_canvas), 'scores': scores, 'n_model_candidates': int(len(scores))}
        _MODEL_CALL_COUNT += 1
        return answer
    finally:
        del result, outputs, inputs, canvas0, canvas1
        if DEVICE.type == 'cuda':
            torch.cuda.empty_cache()

def _texture_std(rgb: np.ndarray, mask: np.ndarray) -> float:
    gray = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY)
    values = gray[mask.astype(bool)]
    return float(values.std()) if values.size else 0.0

def sift_guided_matches(fixed_tile: np.ndarray, moving_tile: np.ndarray, fixed_mask_tile: np.ndarray, moving_mask_tile: np.ndarray) -> tuple[np.ndarray, np.ndarray, np.ndarray, dict[str, int]]:
    if not CFG.ENABLE_SIFT_FALLBACK or not hasattr(cv2, 'SIFT_create'):
        empty = np.empty((0, 2), np.float64)
        return (empty, empty.copy(), np.empty(0), {'sift_raw': 0, 'sift_ratio': 0})
    g0 = cv2.cvtColor(fixed_tile, cv2.COLOR_RGB2GRAY)
    g1 = cv2.cvtColor(moving_tile, cv2.COLOR_RGB2GRAY)
    sift = cv2.SIFT_create(nfeatures=CFG.SIFT_NFEATURES, contrastThreshold=0.01, edgeThreshold=12, sigma=1.4)
    k0, d0 = sift.detectAndCompute(g0, fixed_mask_tile.astype(np.uint8) * 255)
    k1, d1 = sift.detectAndCompute(g1, moving_mask_tile.astype(np.uint8) * 255)
    if d0 is None or d1 is None or len(d0) < 2 or (len(d1) < 2):
        empty = np.empty((0, 2), np.float64)
        return (empty, empty.copy(), np.empty(0), {'sift_raw': 0, 'sift_ratio': 0})
    knn = cv2.BFMatcher(cv2.NORM_L2).knnMatch(d0, d1, k=2)
    good = [a for pair in knn if len(pair) == 2 for a, b in [pair] if a.distance < CFG.SIFT_RATIO_TEST * b.distance]
    if not good:
        empty = np.empty((0, 2), np.float64)
        return (empty, empty.copy(), np.empty(0), {'sift_raw': len(knn), 'sift_ratio': 0})
    p0 = np.asarray([k0[m.queryIdx].pt for m in good], np.float64)
    p1 = np.asarray([k1[m.trainIdx].pt for m in good], np.float64)
    scores = np.asarray([1.0 / (1.0 + m.distance / 100.0) for m in good], np.float64)
    valid = points_inside_mask(fixed_mask_tile, p0) & points_inside_mask(moving_mask_tile, p1)
    return (p0[valid], p1[valid], scores[valid], {'sift_raw': int(len(knn)), 'sift_ratio': int(valid.sum())})

def tile_mutual_matches(fixed_tile: np.ndarray, moving_tile: np.ndarray, fixed_mask_tile: np.ndarray, moving_mask_tile: np.ndarray) -> tuple[np.ndarray, np.ndarray, np.ndarray, dict[str, Any]]:
    forward = infer_correspondences(fixed_tile, moving_tile, canvas=CFG.TILE_MODEL_CANVAS, threshold=CFG.TILE_RAW_MATCH_THRESHOLD)
    f0, f1, fs = (forward['pts0'], forward['pts1'], forward['scores'])
    valid = points_inside_mask(fixed_mask_tile, f0) & points_inside_mask(moving_mask_tile, f1)
    f0, f1, fs = (f0[valid], f1[valid], fs[valid])
    reverse = infer_correspondences(moving_tile, fixed_tile, canvas=CFG.TILE_MODEL_CANVAS, threshold=CFG.TILE_RAW_MATCH_THRESHOLD)
    r1, r0, rs = (reverse['pts0'], reverse['pts1'], reverse['scores'])
    valid_r = points_inside_mask(moving_mask_tile, r1) & points_inside_mask(fixed_mask_tile, r0)
    r0, r1, rs = (r0[valid_r], r1[valid_r], rs[valid_r])
    m0, m1, ms = mutual_consistency_filter(f0, f1, fs, r0, r1, rs, fixed_tile.shape[:2], moving_tile.shape[:2])
    keep = ms >= CFG.TILE_MATCH_SCORE
    m0, m1, ms = (m0[keep], m1[keep], ms[keep])
    method = 'eloftr_mutual'
    sift_counts = {'sift_raw': 0, 'sift_ratio': 0}
    if len(ms) < CFG.TILE_MIN_INLIERS and CFG.ENABLE_SIFT_FALLBACK:
        s0, s1, ss, sift_counts = sift_guided_matches(fixed_tile, moving_tile, fixed_mask_tile, moving_mask_tile)
        if len(ss) > len(ms):
            m0, m1, ms, method = (s0, s1, ss, 'sift_fallback')
    return (m0, m1, ms, {'forward': int(len(forward['scores'])), 'forward_tissue': int(len(fs)), 'reverse': int(len(reverse['scores'])), 'mutual_score': int(keep.sum()), **sift_counts, 'method': method})

def collect_tile_controls(fixed_view: np.ndarray, moving_view: np.ndarray, fixed_mask: np.ndarray, moving_mask: np.ndarray, global_matrix: np.ndarray) -> tuple[np.ndarray, np.ndarray, np.ndarray, pd.DataFrame]:
    global _EMPTY_RELOADS_THIS_CASE
    warped_view = warp_rgb(moving_view, global_matrix, fixed_view.shape[:2])
    warped_mask = warp_mask(moving_mask, global_matrix, fixed_mask.shape)
    overlap = fixed_mask & warped_mask
    selected_tiles = choose_overlap_tiles(overlap)
    inv_global = np.linalg.inv(global_matrix)
    fixed_all, moving_all, scores_all, rows = ([], [], [], [])
    empty_streak = 0
    for rank, (_, x, y) in enumerate(selected_tiles):
        ys = slice(y, min(y + CFG.TILE_SIZE, fixed_view.shape[0]))
        xs = slice(x, min(x + CFG.TILE_SIZE, fixed_view.shape[1]))
        ft, mt = (fixed_view[ys, xs], warped_view[ys, xs])
        fm, mm = (fixed_mask[ys, xs], warped_mask[ys, xs])
        fixed_std, moving_std = (_texture_std(ft, fm), _texture_std(mt, mm))
        accepted = False
        reloaded = False
        try:
            p_fixed, p_warped, scores, counts = tile_mutual_matches(ft, mt, fm, mm)
            both_empty = counts['forward'] == 0 and counts['reverse'] == 0
            empty_streak = empty_streak + 1 if both_empty else 0
            if both_empty and empty_streak >= CFG.AUTO_RELOAD_AFTER_EMPTY_TILES and (_EMPTY_RELOADS_THIS_CASE < CFG.MAX_EMPTY_RELOADS_PER_CASE) and (fixed_std >= 8.0) and (moving_std >= 8.0):
                reload_eloftr(f'consecutive_empty_tiles:rank={rank}')
                _EMPTY_RELOADS_THIS_CASE += 1
                reloaded = True
                p_fixed, p_warped, scores, counts = tile_mutual_matches(ft, mt, fm, mm)
                if counts['forward'] > 0 or counts['reverse'] > 0:
                    empty_streak = 0
            p_fixed += np.array([x, y])
            p_warped += np.array([x, y])
            residual = np.linalg.norm(p_fixed - p_warped, axis=1)
            guided = residual <= CFG.TILE_GUIDED_MAX_RESIDUAL_PX
            p_fixed, p_warped, scores = (p_fixed[guided], p_warped[guided], scores[guided])
            inlier_count = 0
            if len(scores) >= 3:
                local2, inliers = cv2.estimateAffinePartial2D(p_warped.astype(np.float32), p_fixed.astype(np.float32), method=cv2.RANSAC, ransacReprojThreshold=CFG.TILE_RANSAC_THRESHOLD_PX, maxIters=10000, confidence=0.999, refineIters=20)
                if local2 is not None and inliers is not None:
                    keep_local = inliers.reshape(-1).astype(bool)
                    inlier_count = int(keep_local.sum())
                    if inlier_count >= CFG.TILE_MIN_INLIERS:
                        original_moving = transform_points(inv_global, p_warped[keep_local])
                        fixed_all.append(p_fixed[keep_local])
                        moving_all.append(original_moving)
                        scores_all.append(scores[keep_local])
                        accepted = True
            rows.append({'tile_rank': rank, 'x': x, 'y': y, 'width': ft.shape[1], 'height': ft.shape[0], 'overlap_fraction': float((fm & mm).mean()), 'fixed_texture_std': fixed_std, 'moving_texture_std': moving_std, **counts, 'after_guided': int(len(scores)), 'local_inliers': inlier_count, 'accepted': accepted, 'model_reloaded': reloaded, 'error': ''})
        except Exception as exc:
            rows.append({'tile_rank': rank, 'x': x, 'y': y, 'fixed_texture_std': fixed_std, 'moving_texture_std': moving_std, 'accepted': False, 'local_inliers': 0, 'model_reloaded': reloaded, 'error': f'{type(exc).__name__}: {exc}'})
        gc.collect()
        if DEVICE.type == 'cuda':
            torch.cuda.empty_cache()
    if not fixed_all:
        return (np.empty((0, 2)), np.empty((0, 2)), np.empty(0), pd.DataFrame(rows))
    return (np.vstack(fixed_all), np.vstack(moving_all), np.concatenate(scores_all), pd.DataFrame(rows))

def _rgb_uint8(image: np.ndarray) -> np.ndarray:
    arr = np.asarray(image)
    if arr.ndim == 2:
        arr = np.repeat(arr[..., None], 3, axis=2)
    if arr.shape[2] > 3:
        arr = arr[..., :3]
    if arr.dtype != np.uint8:
        lo, hi = np.percentile(arr[np.isfinite(arr)], [0.5, 99.5]) if np.isfinite(arr).any() else (0.0, 1.0)
        arr = np.clip((arr.astype(np.float32) - lo) / max(hi - lo, 1e-06) * 255, 0, 255)
    return arr.astype(np.uint8, copy=False)

def _resize_panel(image: np.ndarray, points: np.ndarray, target_height: int) -> tuple[np.ndarray, np.ndarray]:
    image = _rgb_uint8(image)
    h, w = image.shape[:2]
    scale = min(1.0, target_height / max(h, 1))
    new_w, new_h = (max(1, int(round(w * scale))), max(1, int(round(h * scale))))
    interpolation = cv2.INTER_AREA if scale < 1 else cv2.INTER_LINEAR
    resized = cv2.resize(image, (new_w, new_h), interpolation=interpolation)
    return (resized, np.asarray(points, np.float64).reshape(-1, 2) * scale)

def _stratified_match_indices(inlier_mask: np.ndarray, max_lines: int, *, only_inliers: bool=False) -> np.ndarray:
    mask = np.asarray(inlier_mask, bool).reshape(-1)
    rng = np.random.default_rng(CFG.MATCH_QC_RANDOM_SEED)
    good = np.flatnonzero(mask)
    bad = np.flatnonzero(~mask)
    if only_inliers:
        pool = good
        if len(pool) <= max_lines:
            return pool
        return np.sort(rng.choice(pool, max_lines, replace=False))
    if len(mask) <= max_lines:
        return np.arange(len(mask), dtype=np.int64)
    n_bad = min(len(bad), max(1, int(round(max_lines * 0.4))))
    n_good = min(len(good), max_lines - n_bad)
    remaining = max_lines - n_good - n_bad
    if remaining and len(good) > n_good:
        n_good += min(remaining, len(good) - n_good)
    remaining = max_lines - n_good - n_bad
    if remaining and len(bad) > n_bad:
        n_bad += min(remaining, len(bad) - n_bad)
    chosen_good = rng.choice(good, n_good, replace=False) if n_good else np.empty(0, int)
    chosen_bad = rng.choice(bad, n_bad, replace=False) if n_bad else np.empty(0, int)
    return np.sort(np.r_[chosen_good, chosen_bad].astype(np.int64))

def make_side_by_side_matches(fixed_image: np.ndarray, moving_image: np.ndarray, fixed_points: np.ndarray, moving_points: np.ndarray, inlier_mask: Optional[np.ndarray]=None, *, mode: str='inlier_outlier', left_title: str='H&E fixed', right_title: str='mIF moving') -> np.ndarray:
    p_fixed = np.asarray(fixed_points, np.float64).reshape(-1, 2)
    p_moving = np.asarray(moving_points, np.float64).reshape(-1, 2)
    n = min(len(p_fixed), len(p_moving))
    p_fixed, p_moving = (p_fixed[:n], p_moving[:n])
    mask = np.ones(n, bool) if inlier_mask is None else np.asarray(inlier_mask, bool).reshape(-1)[:n]
    left, p_left = _resize_panel(fixed_image, p_fixed, CFG.MATCH_QC_PANEL_HEIGHT)
    right, p_right = _resize_panel(moving_image, p_moving, CFG.MATCH_QC_PANEL_HEIGHT)
    header, gap = (76, int(CFG.MATCH_QC_GAP_PX))
    canvas_h = header + max(left.shape[0], right.shape[0])
    canvas_w = left.shape[1] + gap + right.shape[1]
    canvas = np.full((canvas_h, canvas_w, 3), 28, np.uint8)
    canvas[header:header + left.shape[0], :left.shape[1]] = left
    x_right = left.shape[1] + gap
    canvas[header:header + right.shape[0], x_right:x_right + right.shape[1]] = right
    cv2.putText(canvas, left_title, (18, 32), cv2.FONT_HERSHEY_SIMPLEX, 0.82, (245, 245, 245), 2, cv2.LINE_AA)
    cv2.putText(canvas, right_title, (x_right + 18, 32), cv2.FONT_HERSHEY_SIMPLEX, 0.82, (245, 245, 245), 2, cv2.LINE_AA)
    only_inliers = mode == 'inliers_only'
    indices = _stratified_match_indices(mask, CFG.MATCH_QC_MAX_LINES, only_inliers=only_inliers)
    overlay = canvas.copy()
    for i in indices:
        a = tuple(np.round(p_left[i] + [0, header]).astype(int))
        b = tuple(np.round(p_right[i] + [x_right, header]).astype(int))
        if mode == 'all':
            color = (35, 190, 255)
        else:
            color = (60, 235, 90) if mask[i] else (255, 70, 70)
        cv2.line(overlay, a, b, color, 1, cv2.LINE_AA)
    canvas = cv2.addWeighted(overlay, float(CFG.MATCH_QC_LINE_ALPHA), canvas, 1.0 - float(CFG.MATCH_QC_LINE_ALPHA), 0)
    for i in indices:
        a = tuple(np.round(p_left[i] + [0, header]).astype(int))
        b = tuple(np.round(p_right[i] + [x_right, header]).astype(int))
        if mode == 'all':
            color = (35, 190, 255)
        else:
            color = (60, 235, 90) if mask[i] else (255, 70, 70)
        cv2.circle(canvas, a, 3, color, -1, cv2.LINE_AA)
        cv2.circle(canvas, b, 3, color, -1, cv2.LINE_AA)
    shown_inliers = int(mask[indices].sum()) if len(indices) else 0
    shown_outliers = int(len(indices) - shown_inliers)
    if mode == 'all':
        legend = f'shown {len(indices)}/{n} controls'
    elif mode == 'inliers_only':
        legend = f'green: affine inliers | shown {shown_inliers}/{int(mask.sum())}'
    else:
        legend = f'green inlier={shown_inliers} | red outlier={shown_outliers} | total controls={n}'
    cv2.putText(canvas, legend, (18, 63), cv2.FONT_HERSHEY_SIMPLEX, 0.58, (230, 230, 230), 1, cv2.LINE_AA)
    if n == 0:
        cv2.putText(canvas, 'No accepted local control points', (18, header + 48), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (255, 90, 90), 2, cv2.LINE_AA)
    return canvas

def save_match_line_qc(pair_dir: Path, fixed_rgb: np.ndarray, moving_rgb: np.ndarray, fixed_view: np.ndarray, moving_view: np.ndarray, affine_warped: np.ndarray, fixed_points: np.ndarray, moving_points: np.ndarray, affine_inliers: np.ndarray, affine_matrix: np.ndarray) -> None:
    fixed_points = np.asarray(fixed_points, np.float64).reshape(-1, 2)
    moving_points = np.asarray(moving_points, np.float64).reshape(-1, 2)
    mask = np.asarray(affine_inliers, bool).reshape(-1)
    if len(mask) != len(fixed_points):
        mask = np.zeros(len(fixed_points), bool)
    save_rgb(pair_dir / 'qc_matches_all.png', make_side_by_side_matches(fixed_rgb, moving_rgb, fixed_points, moving_points, mask, mode='all', left_title='H&E fixed', right_title='mIF moving'))
    save_rgb(pair_dir / 'qc_matches_inlier_outlier.png', make_side_by_side_matches(fixed_rgb, moving_rgb, fixed_points, moving_points, mask, mode='inlier_outlier', left_title='H&E fixed', right_title='mIF moving'))
    save_rgb(pair_dir / 'qc_matches_inliers_only.png', make_side_by_side_matches(fixed_rgb, moving_rgb, fixed_points, moving_points, mask, mode='inliers_only', left_title='H&E fixed', right_title='mIF moving'))
    save_rgb(pair_dir / 'qc_tile_matches_overview.png', make_side_by_side_matches(fixed_view, moving_view, fixed_points, moving_points, mask, mode='inlier_outlier', left_title='H&E matching view', right_title='mIF matching view'))
    affine_predicted = transform_points(affine_matrix, moving_points) if len(moving_points) else np.empty((0, 2), np.float64)
    save_rgb(pair_dir / 'qc_matches_affine_residual.png', make_side_by_side_matches(fixed_rgb, affine_warped, fixed_points, affine_predicted, mask, mode='residual', left_title='H&E fixed', right_title='mIF after affine'))


## Pair registration


In [ ]:
def register_pair_v4(fixed_path: Path, moving_path: Path, cfg: Config=CFG) -> dict[str, Any]:
    started = time.time()
    fixed_case, moving_case = (extract_case_id(fixed_path), extract_case_id(moving_path))
    if fixed_case != moving_case:
        raise ValueError(f'Cross-case registration blocked: mIF {moving_case} -> H&E {fixed_case}')
    case_id = fixed_case
    prepare_matcher_for_case(case_id)
    pair_dir = OUTPUT_DIR / 'pairs' / f'{safe_slug(case_id)}__mIF_to_HE'
    pair_dir.mkdir(parents=True, exist_ok=True)
    fixed_rgb, fixed_meta, _ = read_registration_image(fixed_path, 'HE', cfg)
    moving_rgb, moving_meta, channel_table = read_registration_image(moving_path, 'mIF', cfg)
    if channel_table is not None:
        channel_table.to_csv(pair_dir / 'mif_channel_diagnostics.csv', index=False, encoding='utf-8-sig')
    fixed_mask = tissue_mask(fixed_rgb, 'HE')
    moving_mask = tissue_mask(moving_rgb, 'mIF')
    fixed_view = matching_view(fixed_rgb, fixed_mask, 'HE')
    moving_view = matching_view(moving_rgb, moving_mask, 'mIF')
    save_rgb(pair_dir / 'fixed_registration.png', fixed_rgb)
    save_rgb(pair_dir / 'moving_structural_registration.png', moving_rgb)
    save_rgb(pair_dir / 'fixed_matching_preprocessed.png', fixed_view)
    save_rgb(pair_dir / 'moving_matching_preprocessed.png', moving_view)
    save_rgb(pair_dir / 'fixed_tissue_mask.png', np.repeat((fixed_mask * 255)[..., None], 3, axis=2))
    save_rgb(pair_dir / 'moving_tissue_mask.png', np.repeat((moving_mask * 255)[..., None], 3, axis=2))
    global_matrix, global_info = estimate_global_initial(case_id, fixed_view, moving_view, fixed_mask, moving_mask, pair_dir)
    np.save(pair_dir / 'transform_global_registration.npy', global_matrix)
    global_full = full_resolution_matrix(global_matrix, fixed_meta['low_to_full'], moving_meta['low_to_full'])
    np.save(pair_dir / 'transform_global_full.npy', global_full)
    global_warped = warp_rgb(moving_rgb, global_matrix, fixed_rgb.shape[:2])
    global_mask = warp_mask(moving_mask, global_matrix, fixed_mask.shape)
    global_overlap = tissue_overlap_metrics(fixed_mask, global_mask)
    save_rgb(pair_dir / 'moving_global_warped.png', global_warped)
    save_rgb(pair_dir / 'qc_global_overlay.png', cv2.addWeighted(fixed_rgb, 0.55, global_warped, 0.45, 0.0))
    save_rgb(pair_dir / 'qc_global_contours.png', make_tissue_contour_qc(fixed_rgb, fixed_mask, global_mask))
    tile_fixed, tile_moving, tile_scores, tile_table = collect_tile_controls(fixed_view, moving_view, fixed_mask, moving_mask, global_matrix)
    tile_table.to_csv(pair_dir / 'tile_search_v5.csv', index=False, encoding='utf-8-sig')
    accepted_tiles = int(tile_table.get('accepted', pd.Series(dtype=bool)).fillna(False).astype(bool).sum())
    if len(tile_scores):
        tile_fixed, tile_moving, tile_scores = deduplicate_controls(tile_fixed, tile_moving, tile_scores)
    controls_df = pd.DataFrame({'fixed_x_registration': tile_fixed[:, 0] if len(tile_fixed) else [], 'fixed_y_registration': tile_fixed[:, 1] if len(tile_fixed) else [], 'moving_x_registration': tile_moving[:, 0] if len(tile_moving) else [], 'moving_y_registration': tile_moving[:, 1] if len(tile_moving) else [], 'score': tile_scores})
    refined_matrix, refine_inliers, affine_info = refine_affine(tile_moving, tile_fixed, global_matrix)
    if len(controls_df):
        controls_df['affine_inlier'] = refine_inliers
        global_pred = transform_points(global_matrix, tile_moving)
        refined_pred = transform_points(refined_matrix, tile_moving)
        controls_df['global_residual_px'] = np.linalg.norm(global_pred - tile_fixed, axis=1)
        controls_df['affine_residual_px'] = np.linalg.norm(refined_pred - tile_fixed, axis=1)
    controls_df.to_csv(pair_dir / 'tile_control_points_v5.csv', index=False, encoding='utf-8-sig')
    np.save(pair_dir / 'transform_refined_affine_registration.npy', refined_matrix)
    refined_full = full_resolution_matrix(refined_matrix, fixed_meta['low_to_full'], moving_meta['low_to_full'])
    np.save(pair_dir / 'transform_refined_affine_full.npy', refined_full)
    affine_warped = warp_rgb(moving_rgb, refined_matrix, fixed_rgb.shape[:2])
    affine_mask = warp_mask(moving_mask, refined_matrix, fixed_mask.shape)
    affine_overlap = tissue_overlap_metrics(fixed_mask, affine_mask)
    save_rgb(pair_dir / 'moving_affine_warped.png', affine_warped)
    save_rgb(pair_dir / 'qc_affine_overlay.png', cv2.addWeighted(fixed_rgb, 0.55, affine_warped, 0.45, 0.0))
    save_rgb(pair_dir / 'qc_affine_contours.png', make_tissue_contour_qc(fixed_rgb, fixed_mask, affine_mask))
    save_match_line_qc(pair_dir=pair_dir, fixed_rgb=fixed_rgb, moving_rgb=moving_rgb, fixed_view=fixed_view, moving_view=moving_view, affine_warped=affine_warped, fixed_points=tile_fixed, moving_points=tile_moving, affine_inliers=refine_inliers, affine_matrix=refined_matrix)
    if affine_info.get('accepted', False) and len(refine_inliers) and (int(refine_inliers.sum()) >= CFG.HARD_MIN_INLIERS):
        tps_keep = refine_inliers.copy()
    else:
        residual = np.linalg.norm(transform_points(refined_matrix, tile_moving) - tile_fixed, axis=1) if len(tile_moving) else np.empty(0)
        tps_keep = residual <= CFG.TILE_RANSAC_THRESHOLD_PX
    tps_moving, tps_fixed = (tile_moving[tps_keep], tile_fixed[tps_keep])
    tps_package, rbf = fit_tps_package(tps_moving, tps_fixed, refined_matrix, fixed_meta, moving_meta)
    fold_info = tps_fold_diagnostics(rbf, fixed_mask.shape)
    if fold_info['fold_fraction'] > CFG.MAX_FOLD_FRACTION:
        rbf_for_warp = None
        tps_package['enabled'] = False
        tps_disabled_reason = 'fold_fraction_exceeded'
    else:
        rbf_for_warp = rbf
        tps_disabled_reason = ''
    save_transform_package(pair_dir / 'v5_transform_package.npz', tps_package)
    nonrigid_warped = warp_with_affine_tps(moving_rgb, refined_matrix, rbf_for_warp, fixed_rgb.shape[:2], is_mask=False)
    nonrigid_mask = warp_with_affine_tps(moving_mask.astype(np.uint8), refined_matrix, rbf_for_warp, fixed_mask.shape, is_mask=True)
    nonrigid_overlap = tissue_overlap_metrics(fixed_mask, nonrigid_mask)
    save_rgb(pair_dir / 'moving_nonrigid_warped.png', nonrigid_warped)
    save_rgb(pair_dir / 'qc_nonrigid_overlay.png', cv2.addWeighted(fixed_rgb, 0.55, nonrigid_warped, 0.45, 0.0))
    save_rgb(pair_dir / 'qc_nonrigid_checkerboard.png', make_checkerboard(fixed_rgb, nonrigid_warped))
    save_rgb(pair_dir / 'qc_nonrigid_contours.png', make_tissue_contour_qc(fixed_rgb, fixed_mask, nonrigid_mask))
    if len(tile_fixed):
        predicted = transform_points(refined_matrix, tile_moving)
        if rbf_for_warp is not None:
            predicted += evaluate_tps_displacement(rbf_for_warp, predicted)
        save_rgb(pair_dir / 'qc_control_residuals.png', draw_controls_qc(fixed_rgb, tile_fixed, predicted))
    n_controls = int(len(tile_fixed))
    n_affine_inliers = int(refine_inliers.sum()) if len(refine_inliers) else 0
    affine_ratio = n_affine_inliers / max(n_controls, 1)
    control_coverage = spatial_control_coverage(tile_fixed[tps_keep], fixed_mask.shape) if len(tps_keep) and tps_keep.any() else 0.0
    warnings = []
    if accepted_tiles < CFG.MIN_ACCEPTED_TILES:
        warnings.append(f'accepted_tiles<{CFG.MIN_ACCEPTED_TILES}')
    if n_controls < CFG.MIN_AGGREGATED_CONTROLS:
        warnings.append(f'controls<{CFG.MIN_AGGREGATED_CONTROLS}')
    if not affine_info.get('accepted', False):
        warnings.append('affine_refinement_not_accepted')
    if affine_ratio < CFG.QC_MIN_INLIER_RATIO:
        warnings.append(f'affine_inlier_ratio<{CFG.QC_MIN_INLIER_RATIO}')
    if control_coverage < CFG.QC_MIN_SPATIAL_COVERAGE:
        warnings.append(f'control_coverage<{CFG.QC_MIN_SPATIAL_COVERAGE}')
    if not tps_package['enabled']:
        warnings.append('tps_not_enabled' + (f':{tps_disabled_reason}' if tps_disabled_reason else ''))
    if fold_info['fold_fraction'] > CFG.MAX_FOLD_FRACTION:
        warnings.append('nonrigid_fold_fraction_exceeded')
    hard_fail = n_controls < 6 and global_info.get('source') == 'tissue_mask_pca' or global_overlap['tissue_dice'] < 0.2
    strict_ok = not hard_fail and accepted_tiles >= CFG.MIN_ACCEPTED_TILES and (n_controls >= CFG.MIN_AGGREGATED_CONTROLS) and affine_info.get('accepted', False) and (affine_ratio >= CFG.QC_MIN_INLIER_RATIO) and (control_coverage >= CFG.QC_MIN_SPATIAL_COVERAGE) and bool(tps_package['enabled']) and (fold_info['fold_fraction'] <= CFG.MAX_FOLD_FRACTION)
    status = 'ok' if strict_ok else 'failed' if hard_fail else 'warning'
    quality_score = math.log1p(max(n_affine_inliers, 0)) * max(affine_ratio, 1e-06) * math.sqrt(0.01 + control_coverage) * (0.5 + nonrigid_overlap['tissue_dice'])
    report = {'case_id': case_id, 'status': status, 'provisional_for_manual_tre': status == 'ok', 'accepted_for_cell_level': False, 'manual_tre_required': True, 'warnings': warnings, 'fixed_he': str(fixed_path), 'moving_mif': str(moving_path), 'pairing_verified': True, 'mif_selected_channels': moving_meta.get('mif_selected_channels'), 'global': {'info': global_info, 'tissue_overlap': global_overlap, 'transform_registration': global_matrix, 'transform_full': global_full}, 'tiles': {'attempted': int(len(tile_table)), 'accepted': accepted_tiles, 'aggregated_controls': n_controls}, 'affine': {'info': affine_info, 'inliers': n_affine_inliers, 'inlier_ratio': affine_ratio, 'control_coverage': control_coverage, 'tissue_overlap': affine_overlap, 'transform_registration': refined_matrix, 'transform_full': refined_full}, 'tps': {k: v for k, v in tps_package.items() if k not in {'base_fixed_registration', 'delta_fixed_registration', 'moving_control_registration', 'fixed_control_registration', 'affine_registration', 'fixed_low_to_full', 'moving_full_to_low'}}, 'fold_diagnostics': fold_info, 'nonrigid_tissue_overlap': nonrigid_overlap, 'match_qc_images': ['qc_matches_all.png', 'qc_matches_inlier_outlier.png', 'qc_matches_inliers_only.png', 'qc_tile_matches_overview.png', 'qc_matches_affine_residual.png'], 'quality_score': quality_score, 'fixed_metadata': fixed_meta, 'moving_metadata': moving_meta, 'elapsed_seconds': time.time() - started}
    with (pair_dir / 'metrics_v5.json').open('w', encoding='utf-8') as f:
        json.dump(json_ready(report), f, ensure_ascii=False, indent=2)
    return {'case_id': case_id, 'status': status, 'provisional_for_manual_tre': status == 'ok', 'accepted_for_cell_level': False, 'warnings': ';'.join(warnings), 'pairing_verified': True, 'fixed': fixed_path.name, 'moving': moving_path.name, 'mif_selected_channels': str(moving_meta.get('mif_selected_channels')), 'global_source': global_info.get('source'), 'global_dice': global_overlap['tissue_dice'], 'tiles_attempted': int(len(tile_table)), 'tiles_accepted': accepted_tiles, 'n_controls': n_controls, 'n_inliers': n_affine_inliers, 'n_outliers': n_controls - n_affine_inliers, 'inlier_ratio': affine_ratio, 'control_coverage': control_coverage, 'affine_refined': affine_info.get('accepted', False), 'affine_dice': affine_overlap['tissue_dice'], 'tps_enabled': bool(tps_package['enabled']), 'tps_training_median_px': tps_package.get('training_median_px'), 'tps_training_p95_px': tps_package.get('training_p95_px'), 'fold_fraction': fold_info['fold_fraction'], 'nonrigid_dice': nonrigid_overlap['tissue_dice'], 'quality_score': quality_score, 'elapsed_seconds': report['elapsed_seconds'], 'output_dir': str(pair_dir)}


## Run


In [ ]:
run_info = {'config': asdict(CFG), 'he_fixed_dir': str(HE_DIR), 'mif_moving_dir': str(MIF_DIR), 'v3_results_dir': str(V3_RESULTS_DIR), 'output_dir': str(OUTPUT_DIR), 'device': str(DEVICE), 'torch_version': torch.__version__, 'transformers_version': transformers.__version__, 'pairing_rule': 'strict same case_id; mIF moving -> H&E fixed', 'nonrigid_policy': 'automatic ok is provisional; independent manual TRE required'}
with (OUTPUT_DIR / 'run_config.json').open('w', encoding='utf-8') as f:
    json.dump(json_ready(run_info), f, ensure_ascii=False, indent=2)
summary_rows = []
for fixed_path, moving_path in tqdm(pairs, desc='v4.2 stable + match-line QC mIF -> H&E'):
    case_id = extract_case_id(fixed_path)
    try:
        row = register_pair_v4(fixed_path, moving_path, CFG)
    except Exception as exc:
        pair_dir = OUTPUT_DIR / 'pairs' / f'{safe_slug(case_id)}__mIF_to_HE'
        pair_dir.mkdir(parents=True, exist_ok=True)
        error = {'case_id': case_id, 'status': 'failed', 'fixed': str(fixed_path), 'moving': str(moving_path), 'error_type': type(exc).__name__, 'error': str(exc), 'traceback': traceback.format_exc()}
        (pair_dir / 'error_v5.json').write_text(json.dumps(error, ensure_ascii=False, indent=2), encoding='utf-8')
        row = {'case_id': case_id, 'status': 'failed', 'warnings': '', 'fixed': fixed_path.name, 'moving': moving_path.name, 'pairing_verified': True, 'output_dir': str(pair_dir), 'error': str(exc)}
    summary_rows.append(row)
    gc.collect()
    if DEVICE.type == 'cuda':
        torch.cuda.empty_cache()
summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(OUTPUT_DIR / 'registration_summary_v5.csv', index=False, encoding='utf-8-sig')
display(summary_df)


## QC summary


In [ ]:
if 'summary_df' not in globals() or summary_df.empty:
    path = OUTPUT_DIR / 'registration_summary_v5.csv'
    if not path.exists():
        raise RuntimeError('Run Part 11 first')
    summary_df = pd.read_csv(path)
usable = summary_df[summary_df['status'].isin(['ok', 'warning'])].copy()
if usable.empty:
    pass
else:
    usable['quality_score'] = pd.to_numeric(usable['quality_score'], errors='coerce').fillna(0)
    usable = usable.sort_values(['status', 'quality_score'], ascending=[True, False])
    usable.to_csv(OUTPUT_DIR / 'quality_ranking_v5.csv', index=False, encoding='utf-8-sig')
    columns = ['case_id', 'status', 'warnings', 'mif_selected_channels', 'global_source', 'global_dice', 'tiles_attempted', 'tiles_accepted', 'n_controls', 'n_inliers', 'n_outliers', 'inlier_ratio', 'control_coverage', 'affine_refined', 'affine_dice', 'tps_enabled', 'tps_training_median_px', 'tps_training_p95_px', 'fold_fraction', 'nonrigid_dice', 'quality_score']
    display(usable[[c for c in columns if c in usable.columns]])
    case_to_show = str(usable.iloc[0]['case_id'])
    row = usable[usable['case_id'].astype(str) == case_to_show].iloc[0]
    pair_dir = Path(row['output_dir'])
    images = [('Global contours', pair_dir / 'qc_global_contours.png'), ('Inlier / outlier matches', pair_dir / 'qc_matches_inlier_outlier.png'), ('Inliers only', pair_dir / 'qc_matches_inliers_only.png'), ('Matching-view overview', pair_dir / 'qc_tile_matches_overview.png'), ('Affine residual matches', pair_dir / 'qc_matches_affine_residual.png'), ('TPS/non-rigid contours', pair_dir / 'qc_nonrigid_contours.png'), ('Tile control residuals', pair_dir / 'qc_control_residuals.png'), ('Non-rigid checkerboard', pair_dir / 'qc_nonrigid_checkerboard.png')]
    fig, axes = plt.subplots(4, 2, figsize=(24, 32))
    for ax, (title, image_path) in zip(axes.ravel(), images):
        if image_path.exists():
            ax.imshow(Image.open(image_path))
            ax.set_title(title)
        else:
            ax.text(0.5, 0.5, f'Missing: {image_path.name}', ha='center', va='center')
        ax.axis('off')
    plt.tight_layout()
    plt.show()
